# Creating AI without such libraries as Torch or TensorFlow

In [47]:
import json

save_path = "hackernews_texts.json"
with open(save_path, "r") as f:
    hackernews_texts = json.load(f)

In [48]:
import re

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9 .,!?]", "", text)  # Retain only valid characters
    return text

cleaned_texts = [clean_text(text) for text in hackernews_texts]
concatenated_text = "\n".join(cleaned_texts)

char_list = list()
for letter in concatenated_text:
    char_list.append(letter)

char_list[:10]
vocabulary = sorted(list(set(char_list)))

In [56]:
import numpy as np

stoi = {ch: number for number, ch in enumerate(vocabulary)}
encoded_text = [stoi[ch] for ch in concatenated_text]

tensored_data = np.array(encoded_text)
print(f"Encoded text length: {len(tensored_data)}")

n = int(0.9 * len(tensored_data))
train_data = tensored_data[:n]
val_data = tensored_data[n:]

print(f"Vocabulary length: {len(vocabulary)}")

Encoded text length: 61695
Vocabulary length: 68


In [57]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else val_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

In [60]:
from NoTorchAI.Embedding import Embedding
from NoTorchAI.SGD import ABSGradient


class EmbeddingInitial:
    def __init__(self, vocab_size, d_model, block_size, gradient: ABSGradient):
        self.token_embedding = Embedding(vocab_size, d_model)
        self.position_embedding = Embedding(block_size, d_model)
        self.gradient = gradient

    def _change_weights(self) -> None:
        self.gradient.step(self.token_embedding)
        self.gradient.step(self.position_embedding)

    def forward(self, x):
        B, T = x.shape  

        token_emb = self.token_embedding.forward(x)
        pos_ids = np.arange(T)
        pos_emb = self.position_embedding.forward(pos_ids)

        x = token_emb + pos_emb  

        return x
    
    def backward(self, incoming_grad: np.ndarray) -> None:
        d_token = incoming_grad
        d_pos = np.sum(incoming_grad, axis=0)
        
        self.token_embedding.backward(d_token)
        self.position_embedding.backward(d_pos)

        self._change_weights()

In [ ]:
from NoTorchAI.ActivationFunc import ReLu
from NoTorchAI.Layers.LinearLayer import Linear


class FeedForward:
    def __init__(self, d_model: int, gradient: ABSGradient):
        self.linear1 = Linear(d_model, 4 * d_model)
        self.relu = ReLu()
        self.linear2 = Linear(4 * d_model, d_model)

        self.gradient = gradient

    def _change_weights(self) -> None:
        self.gradient.step(self.linear1)
        self.gradient.step(self.linear2)
        self.gradient.step(self.linear2)

    def forward(self, x):
        linear1 = self.linear1.forward(x)
        relu = self.relu.forward(linear1)
        linear2 = self.linear2.forward(relu)
        return self.net(linear2)
    
    def backward(self, incoming_grad: np.ndarray) -> np.ndarray:
        grad = self.linear2.backward(incoming_grad)
        grad = self.relu.backward(grad)
        grad = self.linear1.backward(grad)
        return grad

In [63]:
from NoTorchAI.SGD import SGD


d_model = 128
vocabulary_size = len(vocabulary)
block_size = 64
batch_size = 32

sgd = SGD(lr=3e-4)
model = EmbeddingInitial(vocab_size=vocabulary_size, d_model=128, block_size=64, gradient=sgd)
xb, yb = get_batch("train", block_size, batch_size)

out = model.forward(xb)
print(out.shape)

(32, 64, 128)
